# Extraction des features — ISMN & Osiris

Télécharge et aligne les **features** pour les datasets ISMN et Osiris :

- **ISMN** (`station_depth_csv`) : météo ERA5 par station, propriétés du sol / topographie, patch DEM, satellites
- **Osiris** (`Osiris_2024` / `Osiris_2025`) : météo par champ, propriétés du sol, satellites
- **Météo locale** Grandvillers (`meteo_locale.csv`)

Prérequis : datasets réorganisés (notebook `csv_for_hrsm`). Les appels GEE réutilisent les fonctions de `Fcn_for_csv.py`.


In [ ]:
import ee
# Authentification GEE (à faire une seule fois)
try:
    ee.Initialize(project="projet-hrms")
    print('GEE déjà initialisé ✓')
except Exception as e:
    print('Initialisation à faire :', e)
    # Dans un terminal : exécuter 'earthengine authenticate' puis relancer
    ee.Authenticate()
    ee.Initialize(project="projet-hrms")
    print('GEE authentifié et initialisé ✓')

# === CONFIG — chemins communs (ISMN + Osiris) ===
BASE_DEST_DIR = r"/home/theodore/Documents/Get_Datasets/station_depth_csv"
INPUT_DIR = BASE_DEST_DIR
UNIFIED = r"/home/theodore/Documents/Get_Datasets/Osiris_data/Osiris_unified"


###  Données Météos

In [ ]:
from Fcn_for_csv import download_station_meteo
download_station_meteo(BASE_DEST_DIR)


# 2. soil properties

In [1]:
import sys
import os
import glob
import pandas as pd
from Fcn_for_csv import get_topo_data

BASE_DEST_DIR = r"/home/theodore/Documents/Get_Datasets/station_depth_csv"
OUTPUT_DEST_DIR = r"/home/theodore/Documents/Get_Datasets/station_depth_csv"
master_path = os.path.join(BASE_DEST_DIR, "Soil_Properties_Master.csv")

get_topo_data(BASE_DEST_DIR, master_path, None)

331 fichiers CSV trouvés. Début de l'extraction des propriétés du sol...

[Sentek-DaD] Nouvelles coordonnées (np.float64(9.793), np.float64(52.9604)) -> Téléchargement & calcul...
[1/331] Sentek-DaD  
[Sentek-DaD] Nouvelles coordonnées (np.float64(13.4771), np.float64(54.3643)) -> Téléchargement & calcul...
[2/331] Sentek-DaD  
[AquaCheck-CP] Nouvelles coordonnées (np.float64(175.9132), np.float64(-40.739)) -> Téléchargement & calcul...
[3/331] AquaCheck-CP  
[Meter-Teros12] Nouvelles coordonnées (np.float64(173.0478), np.float64(-34.7212)) -> Téléchargement & calcul...
[4/331] Meter-Teros12  
[DeltaT-ThetaProbe-ML2x] Nouvelles coordonnées (np.float64(1.6436), np.float64(43.1922)) -> Téléchargement & calcul...
[5/331] DeltaT-ThetaProbe-ML2x  
[DeltaT-ThetaProbe-ML3] Coordonnées (np.float64(1.6436), np.float64(43.1922)) connues -> Récupération depuis le cache.
[6/331] DeltaT-ThetaProbe-ML3  
[AquaCheck-CP] Nouvelles coordonnées (np.float64(175.9019), np.float64(-40.7349)) -> Téléchargem

# Ajout dem manquants

In [ ]:
from Fcn_for_csv import patch_missing_topo
patch_missing_topo(master_path)


# 3. Données satellites

In [ ]:
# Fonctions satellites (déplacées dans Fcn_for_csv.py)
from Fcn_for_csv import (
    bitwiseExtract,
    maskHLSL30,
    maskSentinel2,
    preprocess_vv,
    preprocess_vh,
    merge_bands,
    to_float,
    extract_time_series_to_pandas,
)


In [ ]:
# Fonctions satellites (déplacées dans Fcn_for_csv.py)
from Fcn_for_csv import get_satellite_data_for_point, enrich_csv_with_satellites


In [6]:
import ee
ee.Authenticate()
ee.Initialize()

In [ ]:
from Fcn_for_csv import enrich_stations_with_satellites
enrich_stations_with_satellites(INPUT_DIR)


---

# 📁 SECTION OSIRIS 2025 — Données météo, sol & satellites par CHAMP

Cette section réutilise les fonctions GEE du notebook (météo, sol, satellites) et les applique aux données `Osiris_2025`, groupées par **Champ** (Cressonsacq, Grandvillers, Tarteron).

Structure en entrée : `Osiris_2025/<Champ>/<name>.csv` (sondes après coupe).

En sortie, on ajoute dans **chaque** dossier de champ : `meteo_hourly.csv`, `meteo_daily.csv`, `sentinel_data.csv`, et un `sites_and_soil.csv` à la racine.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, r"/home/theodore/Documents/GitHub/Soil_Moisture")
from Fcn_for_csv import get_meteo_data, get_meteo_data_hourly, get_site_soil_properties_as_dataframe

# ===================== CONFIG ANNÉE =====================
# Choisir l'année à traiter : "2024", "2025" ou "2026"
YEAR = "2026"

# Répertoire source des dossiers par champ (même logique pour 2024/2025/2026)
OSIRIS_BASE = os.path.join(r"/home/theodore/Documents/Get_Datasets/Osiris_data", f"Osiris_{YEAR}")

# Champs à traiter : lus depuis sites.csv (écrit par build_osiris_2024/2025/2026_folders)
sites_csv_path = os.path.join(OSIRIS_BASE, "sites.csv")
sites_df = pd.read_csv(sites_csv_path)
field_centers = {row.site_id: (row.latitude, row.longitude) for row in sites_df.itertuples()}
print(f"{YEAR} : sites.csv = {sites_csv_path}")
print(f"{len(field_centers)} champs :", field_centers)


2026 : sites.csv = /home/theodore/Documents/Get_Datasets/Osiris_data/Osiris_2026/sites.csv
8 champs : {'Champ A': (50.52792828334432, 2.783625013698517), 'Champ B': (50.36857392213168, 1.7288025148262618), 'Champ D': (49.45125663284563, 2.6559426500000005), 'Champ H': (48.32924397764669, 2.199556021035289), 'Champ C': (49.59811879621441, 2.515744570988274), 'Champ G': (48.55376108397808, 3.870774100821636), 'Champ F': (49.264983123021175, 1.456054339547695), 'Champ E': (49.43588760473566, 2.675671624179403)}


In [2]:
import ee
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("\n--- Météo ERA5 par champ ---\n")

for champ, (lat, lon) in field_centers.items():
    dossier_champ = os.path.join(OSIRIS_BASE, champ)
    os.makedirs(dossier_champ, exist_ok=True)

    meteo_hourly_path = os.path.join(dossier_champ, "meteo_hourly.csv")
    meteo_daily_path = os.path.join(dossier_champ, "meteo_daily.csv")

    # Sauter si déjà téléchargé
    if (os.path.isfile(meteo_hourly_path) and os.path.getsize(meteo_hourly_path) > 0
            and os.path.isfile(meteo_daily_path) and os.path.getsize(meteo_daily_path) > 0):
        print(f"[{champ}] déjà traité. Skip.")
        continue

    # Plage temporelle commune à toutes les sondes du champ
    csv_files = sorted(f for f in os.listdir(dossier_champ)
                      if f.endswith('.csv') and f not in
                      ['meteo_hourly.csv', 'meteo_daily.csv', 'sentinel_data.csv'])

    if not csv_files:
        print(f"[{champ}] aucun fichier capteur")
        continue

    global_start, global_end = None, None
    for fname in csv_files:
        df_temp = pd.read_csv(os.path.join(dossier_champ, fname))
        df_temp['timestamp'] = pd.to_datetime(df_temp['timestamp'], format='mixed', utc=True)
        s, e = df_temp['timestamp'].min(), df_temp['timestamp'].max()
        if global_start is None or s < global_start:
            global_start = s
        if global_end is None or e > global_end:
            global_end = e

    print(f"[{champ}] {global_start:%Y-%m-%d} → {global_end:%Y-%m-%d}")

    # Horaire
    df_meteo_hourly = get_meteo_data_hourly(lat, lon, global_start, global_end)
    if df_meteo_hourly.empty:
        print(f"  [{champ}] pas de données ERA5 horaires")
    else:
        df_meteo_hourly.to_csv(meteo_hourly_path, index=True, index_label='timestamp')
    
    # Quotidien
    df_meteo_daily = get_meteo_data(lat, lon, global_start, global_end)
    if df_meteo_daily.empty:
        print(f"  [{champ}] pas de données ERA5 journalières")
    else:
        df_meteo_daily.to_csv(meteo_daily_path, index=True, index_label='timestamp')
    
    print(f"  ✓ [{champ}] hourly ({len(df_meteo_hourly)}), daily ({len(df_meteo_daily)})")


--- Météo ERA5 par champ ---

[Champ A] 2026-05-05 → 2026-09-09
  ✓ [Champ A] hourly (2922), daily (121)
[Champ B] 2026-05-05 → 2026-08-25
  ✓ [Champ B] hourly (2688), daily (112)
[Champ D] 2026-04-28 → 2026-08-04
  ✓ [Champ D] hourly (2352), daily (98)
[Champ H] 2026-05-25 → 2026-08-31
  ✓ [Champ H] hourly (2352), daily (98)
[Champ C] 2026-05-23 → 2026-08-24
  ✓ [Champ C] hourly (2232), daily (93)
[Champ G] 2026-05-23 → 2026-08-22
  ✓ [Champ G] hourly (2184), daily (91)
[Champ F] 2026-05-23 → 2026-08-03
  ✓ [Champ F] hourly (1728), daily (72)
[Champ E] 2026-06-24 → 2026-08-08
  ✓ [Champ E] hourly (1080), daily (45)


In [3]:
print("\n--- Propriétés du sol par champ ---\n")

all_results = []
for champ, (lat, lon) in field_centers.items():
    print(f"[{champ}], Coordonnées {(lat, lon)} -> Téléchargement & calcul...")
    df_props = get_site_soil_properties_as_dataframe(
        site_id=champ,
        longitude=lon,
        latitude=lat,
        soil_types=["bulk", "silt", "clay", "sand", "Ksat", "dem"],
        verbose=False
    )
    all_results.append(df_props)

df_all_sites_soil = pd.concat(all_results, ignore_index=True)
master_output = os.path.join(OSIRIS_BASE, "sites_and_soil.csv")
df_all_sites_soil.to_csv(master_output, index=False)
print(f"\nExtraction terminée ! Fichier maître sauvegardé ici : {master_output}")
print(df_all_sites_soil)


--- Propriétés du sol par champ ---

[Champ A], Coordonnées (50.52792828334432, 2.783625013698517) -> Téléchargement & calcul...
[Champ B], Coordonnées (50.36857392213168, 1.7288025148262618) -> Téléchargement & calcul...
[Champ D], Coordonnées (49.45125663284563, 2.6559426500000005) -> Téléchargement & calcul...
[Champ H], Coordonnées (48.32924397764669, 2.199556021035289) -> Téléchargement & calcul...
[Champ C], Coordonnées (49.59811879621441, 2.515744570988274) -> Téléchargement & calcul...
[Champ G], Coordonnées (48.55376108397808, 3.870774100821636) -> Téléchargement & calcul...
[Champ F], Coordonnées (49.264983123021175, 1.456054339547695) -> Téléchargement & calcul...
[Champ E], Coordonnées (49.43588760473566, 2.675671624179403) -> Téléchargement & calcul...

Extraction terminée ! Fichier maître sauvegardé ici : /home/theodore/Documents/Get_Datasets/Osiris_data/Osiris_2026/sites_and_soil.csv
   site_id  longitude   latitude  dem_m_30m_depth  bulk_m_30m_0cm_30cm  \
0  Champ A   

/tmp/ipykernel_33640/150313041.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_sites_soil = pd.concat(all_results, ignore_index=True)


In [4]:
from Fcn_for_csv import get_satellite_data_for_point

In [5]:
print("\n--- Données satellites par champ ---\n")

for champ, (lat, lon) in field_centers.items():
    dossier_champ = os.path.join(OSIRIS_BASE, champ)
    sentinel_path = os.path.join(dossier_champ, "sentinel_data.csv")

    # Sauter si déjà téléchargé
    if os.path.isfile(sentinel_path) and os.path.getsize(sentinel_path) > 0:
        print(f"[{champ}] déjà traité. Skip.")
        continue

    csv_files = sorted(f for f in os.listdir(dossier_champ)
                      if f.endswith('.csv') and f not in
                      ['meteo_hourly.csv', 'meteo_daily.csv', 'sentinel_data.csv'])
    if not csv_files:
        print(f"[{champ}] aucun fichier capteur")
        continue

    global_start, global_end = None, None
    for fname in csv_files:
        df_temp = pd.read_csv(os.path.join(dossier_champ, fname))
        df_temp['timestamp'] = pd.to_datetime(df_temp['timestamp'], format='mixed', utc=True)
        s, e = df_temp['timestamp'].min(), df_temp['timestamp'].max()
        if global_start is None or s < global_start:
            global_start = s
        if global_end is None or e > global_end:
            global_end = e

    print(f"[{champ}] {global_start:%Y-%m-%d} → {global_end:%Y-%m-%d}")

    df_sat = get_satellite_data_for_point(lon, lat, global_start, global_end)
    if df_sat.empty:
        print(f"  [{champ}] pas de données satellites")
        continue

    df_sat.to_csv(sentinel_path, index=True, index_label='timestamp')
    print(f"  ✓ [{champ}] satellite ({len(df_sat)} lignes)")


--- Données satellites par champ ---

[Champ A] 2026-05-05 → 2026-09-09
  ✓ [Champ A] satellite (113 lignes)
[Champ B] 2026-05-05 → 2026-08-25
  ✓ [Champ B] satellite (99 lignes)
[Champ D] 2026-04-28 → 2026-08-04
  ✓ [Champ D] satellite (87 lignes)
[Champ H] 2026-05-25 → 2026-08-31
  ✓ [Champ H] satellite (85 lignes)
[Champ C] 2026-05-23 → 2026-08-24
  ✓ [Champ C] satellite (76 lignes)
[Champ G] 2026-05-23 → 2026-08-22
  ✓ [Champ G] satellite (76 lignes)
[Champ F] 2026-05-23 → 2026-08-03
  ✓ [Champ F] satellite (64 lignes)
[Champ E] 2026-06-24 → 2026-08-08
  ✓ [Champ E] satellite (38 lignes)


---
# 📁 SECTION MÉTÉO LOCALE — `meteo_locale.csv` (station Grandvillers)

Crée `meteo_locale.csv`, série journalière de la **station météo locale** (embarquée
dans les fichiers sondes bruts, colonnes `IRRAD, TMIN, TMAX, VAP, WIND, RAIN, irrig_mm`),
dans le même format que le `meteo_daily.csv` (ERA5).

La météo de site est identique pour toutes les sondes : on concatène les fichiers de
`Grandvillers_data/raw`, on déduplique par date.

Écrit dans deux endroits (même contenu, colonne temporelle adaptée) :
- `Grandvillers_data/meteo_locale.csv` → colonne `date`
- `Osiris_unified/2025_Grandvillers/meteo_locale.csv` → colonne `timestamp` (prêt à être joint par `get_osiris_data`)


In [ ]:
from Fcn_for_csv import create_meteo_locale

GV_BASE = r"/home/theodore/Documents/Get_Datasets/Grandvillers_data"

df_meteo_locale = create_meteo_locale(
    raw_dir=os.path.join(GV_BASE, "raw"),
    dest_grandvillers=os.path.join(GV_BASE, "meteo_locale.csv"),
    dest_unified=os.path.join(UNIFIED, "2025_Grandvillers", "meteo_locale.csv"),
)